# Text-to-SQL: QLoRA-дообучение Qwen2.5-1.5B на бесплатной Colab T4

Ноутбук проходит весь цикл проекта:

1. подготовка данных (sql-create-context);
2. оценка **базовой** модели на своём бенчмарке и held-out выборке;
3. QLoRA-обучение (ориентировочно 40–90 минут на T4 для 15 000 примеров);
4. оценка **дообученной** модели;
5. (опционально) оценка API-модели для сравнения по качеству и цене;
6. сводная таблица, график и разбор ошибок.

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Код и зависимости
Замени `YOUR_USERNAME` на свой логин GitHub (после того как загрузишь репозиторий).

In [ ]:
REPO = "https://github.com/sofetizm/text2sql-lora.git"
!git clone -q $REPO
%cd text2sql-lora
!pip install -q -r requirements.txt && pip install -q -e .

### (Опционально) Google Drive
Colab стирает файлы после отключения. Чтобы адаптер и результаты не потерялись, подключи Drive —
в конце ноутбука они туда скопируются.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/text2sql-lora'

## 2. Данные

In [ ]:
!python -m t2s.data --out data
!python -m t2s.benchmark_db --out benchmark/dbs

## 3. Базовая модель (до обучения)
Обе модели оцениваются в одинаковой 4-битной квантизации — иначе сравнение нечестное.

In [ ]:
!python -m t2s.evaluate --name base --backend hf --model Qwen/Qwen2.5-1.5B-Instruct --load-in-4bit

## 4. Обучение
Для быстрой проверки, что всё работает, добавь `--set training.max_steps=50`.

In [ ]:
!python -m t2s.train --config configs/qlora_qwen1.5b.yaml

## 5. Дообученная модель

In [ ]:
!python -m t2s.evaluate --name lora --backend hf --model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter outputs/qwen1.5b-sql-lora/adapter --load-in-4bit

## 6. (Опционально) API-модель для сравнения
Подходит любое OpenAI-совместимое API. Ключ положи в секреты Colab (значок ключа слева)
под именем `OPENAI_API_KEY`; для другого провайдера добавь секрет `OPENAI_BASE_URL`.
Цены за 1 млн токенов возьми со страницы тарифов провайдера.

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
try:
    os.environ['OPENAI_BASE_URL'] = userdata.get('OPENAI_BASE_URL')
except Exception:
    pass

API_MODEL = 'MODEL_NAME'   # имя модели у провайдера
PRICE_IN, PRICE_OUT = 0.0, 0.0   # USD за 1 млн входных / выходных токенов
!python -m t2s.evaluate --name api --backend openai --model {API_MODEL} \
    --price-in {PRICE_IN} --price-out {PRICE_OUT}

## 7. Итоги

In [ ]:
!python -m t2s.report --results results
!python -m t2s.compare results/base_predictions.jsonl results/lora_predictions.jsonl

from IPython.display import Markdown, Image, display
display(Markdown(open('results/RESULTS.md').read().replace('![accuracy](accuracy.png)', '')))
display(Image('results/accuracy.png'))

## 8. Сохранение
Скопируй папку `results/` в репозиторий и закоммить — таблица и график пойдут в README.

In [ ]:
!mkdir -p "$SAVE_DIR" && cp -r results outputs/qwen1.5b-sql-lora/adapter outputs/qwen1.5b-sql-lora/train_meta.json "$SAVE_DIR"/
!cd results && zip -qr ../results.zip . && cd ..
from google.colab import files
files.download('results.zip')

### (Опционально) Загрузить адаптер на Hugging Face Hub
Тогда демо `app.py` можно запустить в Hugging Face Space, указав `ADAPTER=<твой-логин>/qwen1.5b-text2sql-lora`.

In [ ]:
from huggingface_hub import create_repo, login, upload_folder
login()
HF_REPO = 'YOUR_HF_USERNAME/qwen1.5b-text2sql-lora'
create_repo(HF_REPO, exist_ok=True)
upload_folder(repo_id=HF_REPO, folder_path='outputs/qwen1.5b-sql-lora/adapter')